# R1-05 · SQL para análisis — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R1 · *Analista de Datos* · Semana 6 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Consultar una base con **SELECT / FROM / WHERE**.
2. Armar rankings con **ORDER BY** y **LIMIT**.
3. Resumir con **GROUP BY** y filtrar grupos con **HAVING**.
4. Cruzar tablas con **JOIN** y auditar con **LEFT JOIN ... IS NULL**.
5. Combinar SQL y pandas para interpretar **promedio vs mediana**.

**Competencia de salida:** consultar una base de datos relacional con SQL para responder preguntas de gestión.

**Dato real:** matrícula 2025 por RBD × directorio oficial de establecimientos (MINEDUC), en SQLite.

**Entregable:** que las **8 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request, sqlite3
import pandas as pd

# "En vivo o caché": usa los archivos locales; si faltan (ej. Colab), se bajan del repo.
for archivo in ["matricula_rbd.csv", "directorio_establecimientos.csv"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{archivo}", archivo)

df_matricula = pd.read_csv("matricula_rbd.csv")                 # cuántos estudiantes (por RBD)
df_directorio = pd.read_csv("directorio_establecimientos.csv")  # quién es cada RBD

# Una base de datos SQLite en memoria con DOS tablas
conn = sqlite3.connect(":memory:")
df_matricula.to_sql("matricula", conn, index=False, if_exists="replace")
df_directorio.to_sql("directorio", conn, index=False, if_exists="replace")

def consultar(sql):
    """Ejecuta una consulta SQL y devuelve el resultado como tabla de pandas."""
    return pd.read_sql_query(sql, conn)

print("Tablas en la base:", list(consultar("SELECT name FROM sqlite_master WHERE type='table'")["name"]))
consultar("SELECT * FROM directorio LIMIT 5")

## ¿Por qué SQL?

Hasta ahora traías **toda** la tabla a Python y la filtrabas con pandas. Los sistemas del Estado
(SIGE del MINEDUC, registros de salud, el Registro Social de Hogares) no viven en un CSV: viven en
**bases de datos** con millones de filas, y se consultan con **SQL**. SQL es la «boleta de pedido»
que le entregas a la base: describes **qué** quieres y ella te devuelve **solo eso**.

Usaremos **SQLite**, que viene incluido en Python (`sqlite3`): cero instalación, funciona en Colab.
La base tiene dos tablas, igual que en R1-03:

| Tabla | Filas | Columnas |
|---|---|---|
| `matricula` | 10.949 | `rbd`, `mat_total`, `mat_hombres`, `mat_mujeres` |
| `directorio` | 10.945 | `rbd`, `nombre`, `dependencia`, `rural` (1 = rural), `region`, `comuna` |

Toda consulta básica tiene esta forma (por costumbre, las palabras clave en MAYÚSCULAS):

```sql
SELECT columnas
FROM   tabla
WHERE  condición          -- filtra filas
GROUP BY columna          -- agrupa
HAVING condición          -- filtra grupos
ORDER BY columna DESC     -- ordena
LIMIT  n                  -- corta
```

> En cada ejercicio guardas tu consulta **como texto** en una variable (`sql_01`, `sql_02`, ...)
> y la ejecutas con `consultar(...)`.

## 1. SELECT y FROM: elegir columnas

`SELECT` dice **qué columnas**; `FROM`, de **qué tabla**. `SELECT *` trae todas. Es el equivalente a `df[["nombre", "comuna"]]`, pero pidiéndoselo a la base.

In [ ]:
consultar("SELECT nombre, region FROM directorio LIMIT 3")

### ✍️ Ejercicio 1 — Tu primer SELECT

In [ ]:
sql_01 = "SELECT nombre, comuna, dependencia FROM directorio"
r = consultar(sql_01)
print(r.shape)
r.head()

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    r = consultar(sql_01)
    assert list(r.columns) == ["nombre", "comuna", "dependencia"], f"Columnas obtenidas: {list(r.columns)}"
    assert len(r) == len(df_directorio), "Sin WHERE ni LIMIT deberías traer todas las filas"
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'SELECT nombre, comuna, dependencia FROM directorio')
    print("   Detalle:", e)

## 2. WHERE: filtrar filas

`WHERE` se queda con las filas que cumplen una condición. Operadores: `=` (¡uno solo, no `==`!),
`<>` (distinto), `>`, `<`, `>=`, `<=`, y se combinan con `AND` / `OR`. **Los textos van entre
comillas simples**: `WHERE region = 'Maule'`.

Pregunta real: la Seremi de La Araucanía planifica transporte escolar y quiere la lista de
**establecimientos rurales** de su región.

### ✍️ Ejercicio 2 — Los establecimientos rurales de La Araucanía

In [ ]:
sql_02 = """
SELECT nombre, comuna
FROM directorio
WHERE region = 'La Araucanía' AND rural = 1
"""
r = consultar(sql_02)
print("establecimientos rurales en La Araucanía:", len(r))
r.head()

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    r = consultar(sql_02)
    _esp = df_directorio[(df_directorio["region"] == "La Araucanía") & (df_directorio["rural"] == 1)]
    assert len(r) == len(_esp), f"Esperaba {len(_esp)} filas, obtuve {len(r)}"
    assert list(r.columns) == ["nombre", "comuna"]
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "WHERE region = 'La Araucanía' AND rural = 1  (comillas simples y tilde).")
    print("   Detalle:", e)

## 3. ORDER BY y LIMIT: rankings

`ORDER BY columna DESC` ordena de mayor a menor (`ASC`, el defecto, de menor a mayor). `LIMIT n` se queda con las primeras `n` filas. Juntos arman un **ranking**: ¿cuáles son los 5 establecimientos con más estudiantes del país?

### ✍️ Ejercicio 3 — Los 5 establecimientos más grandes

In [ ]:
sql_03 = "SELECT rbd, mat_total FROM matricula ORDER BY mat_total DESC LIMIT 5"
consultar(sql_03)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    r = consultar(sql_03)
    assert len(r) == 5, "Usa LIMIT 5"
    assert list(r["mat_total"]) == sorted(df_matricula["mat_total"], reverse=True)[:5], "Ordena con ORDER BY mat_total DESC"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'SELECT rbd, mat_total FROM matricula ORDER BY mat_total DESC LIMIT 5')
    print("   Detalle:", e)

In [ ]:
# Ojo: solo vemos el RBD, no el nombre. Para ponerle nombre hay que CRUZAR con el directorio
# → lo harás con JOIN en el ejercicio 6.

## 4. GROUP BY: resumir por grupo

Las **funciones de agregación** convierten muchas filas en un número: `COUNT(*)`, `SUM(col)`,
`AVG(col)`, `MIN(col)`, `MAX(col)`. Con `GROUP BY` se aplican **a cada grupo** por separado (la
tabla dinámica de Excel). `AS` le pone nombre a la columna resultante.

> **Regla de oro:** en el `SELECT` solo van las columnas del `GROUP BY` y funciones de agregación.

### ✍️ Ejercicio 4 — Establecimientos por dependencia

In [ ]:
sql_04 = """
SELECT dependencia, COUNT(*) AS n
FROM directorio
GROUP BY dependencia
ORDER BY n DESC
"""
consultar(sql_04)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    r = consultar(sql_04)
    _esp = df_directorio["dependencia"].value_counts()
    assert list(r.columns) == ["dependencia", "n"], "Columnas: dependencia y n (COUNT(*) AS n)"
    assert list(r["n"]) == list(_esp.values), "Revisa el conteo y el orden (ORDER BY n DESC)"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'SELECT dependencia, COUNT(*) AS n FROM directorio GROUP BY dependencia ORDER BY n DESC')
    print("   Detalle:", e)

## 5. HAVING: filtrar grupos (no filas)

`WHERE` filtra **filas** antes de agrupar; `HAVING` filtra **grupos** después de agrupar.
Por eso `WHERE COUNT(*) >= 40` da error (el conteo todavía no existe) y `HAVING COUNT(*) >= 40` funciona.

Pregunta real: ¿en qué **comunas** se concentra la educación rural? Busca las comunas con **al
menos 40 establecimientos rurales** (pistas para priorizar programas de escuelas multigrado y
transporte escolar).

### ✍️ Ejercicio 5 — Comunas con 40 o más establecimientos rurales

In [ ]:
sql_05 = """
SELECT comuna, COUNT(*) AS n_rurales
FROM directorio
WHERE rural = 1
GROUP BY comuna
HAVING COUNT(*) >= 40
ORDER BY n_rurales DESC
"""
consultar(sql_05)

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    r = consultar(sql_05)
    _c = df_directorio[df_directorio["rural"] == 1].groupby("comuna").size()
    _esp = _c[_c >= 40].sort_values(ascending=False)
    assert set(r["comuna"]) == set(_esp.index), f"Esperaba {len(_esp)} comunas: revisa WHERE rural = 1 y HAVING COUNT(*) >= 40"
    assert int(r["n_rurales"].min()) >= 40
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'WHERE rural = 1 (filtra filas) ... HAVING COUNT(*) >= 40 (filtra grupos).')
    print("   Detalle:", e)

## 6. JOIN: cruzar dos tablas

`JOIN ... ON` cruza dos tablas por la llave, igual que `pd.merge` en R1-03. Se usan **alias**
cortos (`m`, `d`) para decir de qué tabla viene cada columna:

```sql
SELECT d.region, SUM(m.mat_total) AS estudiantes
FROM matricula AS m
JOIN directorio AS d ON m.rbd = d.rbd
GROUP BY d.region
```

Pregunta real: ¿cuántos estudiantes hay en cada región, de mayor a menor?

### ✍️ Ejercicio 6 — Estudiantes por región con JOIN

In [ ]:
sql_06 = """
SELECT d.region, SUM(m.mat_total) AS estudiantes
FROM matricula AS m
JOIN directorio AS d ON m.rbd = d.rbd
GROUP BY d.region
ORDER BY estudiantes DESC
"""
r = consultar(sql_06)
r.head()

In [ ]:
# ── Celda de chequeo · Ejercicio 6 ──────────────────────────────────────────
try:
    r = consultar(sql_06)
    _esp = pd.merge(df_matricula, df_directorio, on="rbd").groupby("region")["mat_total"].sum().sort_values(ascending=False)
    assert len(r) == len(_esp), "Debe haber una fila por región"
    assert list(r["region"]) == list(_esp.index), "Revisa el orden (ORDER BY estudiantes DESC)"
    assert list(r["estudiantes"]) == list(_esp.values)
    print("✅ Ejercicio 6: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'JOIN directorio AS d ON m.rbd = d.rbd ... ORDER BY estudiantes DESC')
    print("   Detalle:", e)

## 7. LEFT JOIN: auditar lo que no calza

El `JOIN` normal (inner) **bota** las filas sin pareja. `LEFT JOIN` las conserva y deja en
`NULL` las columnas de la otra tabla. En SQL un nulo se pregunta con `IS NULL` (nunca con `= NULL`).

Es la misma auditoría de R1-03, ahora en SQL: ¿qué RBD reportan matrícula pero **no están** en
el directorio oficial?

### ✍️ Ejercicio 7 — Encuentra los RBD huérfanos

In [ ]:
sql_07 = """
SELECT m.rbd, m.mat_total
FROM matricula AS m
LEFT JOIN directorio AS d ON m.rbd = d.rbd
WHERE d.rbd IS NULL
"""
consultar(sql_07)

In [ ]:
# ── Celda de chequeo · Ejercicio 7 ──────────────────────────────────────────
try:
    r = consultar(sql_07)
    _esp = set(df_matricula["rbd"]) - set(df_directorio["rbd"])
    assert set(r["rbd"]) == _esp, f"Esperaba {len(_esp)} RBD huérfanos: usa WHERE d.rbd IS NULL"
    print("✅ Ejercicio 7: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'WHERE d.rbd IS NULL  (los que el LEFT JOIN dejó sin pareja).')
    print("   Detalle:", e)

## 8. Interpretar: ¿promedio o mediana?

SQL calcula el promedio con `AVG`, pero SQLite **no tiene mediana**. Lo práctico: SQL trae
los datos justos y pandas termina el cálculo. Compara el tamaño de los establecimientos
**municipales** y **particulares subvencionados** (las dos dependencias más grandes del sistema):

1. `promedios`: con SQL, el promedio de `mat_total` por dependencia (`AVG`, con JOIN).
2. `medianas`: con pandas, la mediana por dependencia sobre una consulta que traiga
   `dependencia` y `mat_total`.
3. Elige la interpretación correcta en `conclusion` (`"A"`, `"B"` o `"C"`):

- **A.** El particular subvencionado típico es mucho más grande que el municipal típico: el
  promedio y la mediana dicen lo mismo.
- **B.** El promedio exagera la diferencia: unos pocos particulares subvencionados muy grandes
  inflan su promedio; con la **mediana**, el establecimiento típico de ambas dependencias tiene
  un tamaño parecido.
- **C.** Los municipales son más grandes que los particulares subvencionados.

### ✍️ Ejercicio 8 — Promedio (SQL) vs mediana (pandas)

In [ ]:
promedios = consultar("""
SELECT d.dependencia, AVG(m.mat_total) AS promedio
FROM matricula AS m JOIN directorio AS d ON m.rbd = d.rbd
GROUP BY d.dependencia
""").set_index("dependencia")["promedio"]

datos = consultar("""
SELECT d.dependencia, m.mat_total
FROM matricula AS m JOIN directorio AS d ON m.rbd = d.rbd
""")
medianas = datos.groupby("dependencia")["mat_total"].median()

comparar = pd.DataFrame({"promedio": promedios.round(0), "mediana": medianas})
print(comparar.loc[["Municipal", "Particular subvencionado"]])
conclusion = "B"

In [ ]:
# ── Celda de chequeo · Ejercicio 8 ──────────────────────────────────────────
try:
    _x = pd.merge(df_matricula, df_directorio, on="rbd")
    _prom = _x.groupby("dependencia")["mat_total"].mean()
    _med = _x.groupby("dependencia")["mat_total"].median()
    assert abs(promedios["Municipal"] - _prom["Municipal"]) < 0.01, "Revisa el AVG por dependencia"
    assert abs(medianas["Particular subvencionado"] - _med["Particular subvencionado"]) < 0.01, "Revisa la mediana por dependencia"
    _r_prom = _prom["Particular subvencionado"] / _prom["Municipal"]
    _r_med = _med["Particular subvencionado"] / _med["Municipal"]
    _letra = "C" if _r_prom < 1 else ("B" if _r_med < 1.15 and _r_prom > 1.3 else "A")
    assert str(conclusion).strip().upper() == _letra, "Compara la razón entre dependencias usando el promedio y usando la mediana"
    print("✅ Ejercicio 8: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "promedios: AVG(m.mat_total); medianas: datos.groupby('dependencia')['mat_total'].median(). Luego compara.")
    print("   Detalle:", e)

## ✍️ Reflexión (no se corrige)

Un titular dice: «Los colegios particulares subvencionados tienen 44% más estudiantes que los
municipales». ¿Es falso? ¿Es engañoso? Escribe en una frase cómo lo reformularías con lo que
viste en el ejercicio 8.

## Errores típicos en SQL

- **Texto sin comillas** → `WHERE region = Maule` falla; es `WHERE region = 'Maule'`.
- **`==` en vez de `=`** → en SQL la igualdad es un solo `=`.
- **`= NULL`** → nunca es verdadero; se pregunta con `IS NULL` / `IS NOT NULL`.
- **Filtrar un conteo con WHERE** → `WHERE COUNT(*) > 40` falla; los grupos se filtran con `HAVING`.
- **Columna ambigua en un JOIN** → si `rbd` está en ambas tablas, di de cuál: `m.rbd`.
- **Esperar orden sin pedirlo** → sin `ORDER BY`, la base devuelve en cualquier orden.

## Cierre

- `SELECT/FROM` elige, `WHERE` filtra filas, `GROUP BY` resume, `HAVING` filtra grupos,
  `ORDER BY/LIMIT` rankea.
- `JOIN` cruza por la llave; `LEFT JOIN ... WHERE ... IS NULL` audita lo que no calza.
- SQL trae lo justo; pandas termina lo que SQL no sabe hacer (como la mediana).

> *En R1-06 vas a mirar esos promedios y medianas con lupa: estadística descriptiva para no
> dejarte engañar por un solo número.*